# Food Detection with YOLO (UEC FOOD 100 subset)

主食・主菜・副菜・汁物の12品目を対象に、YOLOで複数品目の物体検出を行います。
分類演習(`food_CNN_classification.ipynb`)が「1品目だけ切り出した画像」を扱ったのに対し、
こちらは給食トレイのような**1枚の写真に複数品目が写っている**、より実際の場面に近い画像を扱います。

事前準備:
- `convert_uecfood100_to_yolo.py` と `category_ja_utf8.txt` をGoogleドライブに配置しておく
  (生データ本体はこのノートブック内で直接ダウンロードするため、アップロード不要)


## 1. Googleドライブのマウントと生データのダウンロード

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!rm -rf /content/UECFOOD100
!wget -q http://foodcam.mobi/dataset100.zip -O /content/dataset100.zip
!unzip -q -o /content/dataset100.zip -d /content/

In [ ]:
base_dir = '/content/drive/MyDrive/Colab Notebooks/food_yolo/'
dataset_root = '/content/UECFOOD100/'
output_dir = base_dir + 'yolo_data/'
category_names_path = base_dir + 'category_ja_utf8.txt'
convert_script_path = base_dir + 'convert_uecfood100_to_yolo.py'

## 2. ライブラリのインストール

In [ ]:
!pip install ultralytics pillow -q

## 3. データ変換(bb_info.txt → YOLO形式)

以前用意した `convert_uecfood100_to_yolo.py` を、選んだ12品目を指定して実行します。
同じ写真が複数カテゴリに重複している場合(例: ごはん+味噌汁)も、自動的に1枚の画像・
1つのラベルファイルにまとめられます。

In [ ]:
CATEGORIES = "1,36,46,55,56,60,63,67,69,70,87,90"

import os

# convert_script_path が存在するか確認
if not os.path.exists(convert_script_path):
    print(f"エラー: スクリプトファイル '{convert_script_path}' が見つかりません。Googleドライブにアップロードされているか確認してください。")
else:
    # コマンド文字列をPythonで明示的に構築し、変数が正しく展開されるようにする
    command = f"python \"{convert_script_path}\" " \
              f"--dataset-root \"{dataset_root}\" " \
              f"--output-dir \"{output_dir}\" " \
              f"--categories \"{CATEGORIES}\" " \
              f"--category-names \"{category_names_path}\" " \
              f"--val-fraction 0.2 " \
              f"--seed 42"
    !{command}

In [ ]:
# 出力内容の確認
import os

for split in ("train", "val"):
    # フォルダが存在しない場合にエラーにならないよう、ディレクトリをチェック
    images_path = output_dir + f"images/{split}"
    labels_path = output_dir + f"labels/{split}"

    n_images = len(os.listdir(images_path)) if os.path.exists(images_path) else 0
    n_labels = len(os.listdir(labels_path)) if os.path.exists(labels_path) else 0

    print(f"{split}: images={n_images}, labels={n_labels}")

print()
# output_dirにdata.yamlが存在するかチェック
data_yaml_path = output_dir + "data.yaml"
if os.path.exists(data_yaml_path):
    with open(data_yaml_path, encoding="utf-8") as f:
        print(f.read())
else:
    print(f"エラー: data.yaml が '{data_yaml_path}' に見つかりません。")

## 4. 学習(Fine-tuning)

In [ ]:
from ultralytics import YOLO

# nano(最小・最速)モデルをベースに、今回の12クラスでfine-tuning
model = YOLO("yolo11n.pt")

results = model.train(
    data=output_dir + "data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    name="food_yolo",
)

## 5. 評価(mAP・Precision・Recallなど)

検証データで、学習したモデルの指標を確認します。


In [ ]:
metrics = model.val()

print(f"mAP@50:    {metrics.box.map50:.3f}")
print(f"mAP@50-95: {metrics.box.map:.3f}")
print(f"Precision: {metrics.box.mp:.3f}")
print(f"Recall:    {metrics.box.mr:.3f}")

## 6. 推論と可視化

学習したモデルで、実際の写真から複数品目を検出してみます。


In [ ]:
import matplotlib.pyplot as plt

def detect_and_show(image_path, confidence=0.4):
    result = model.predict(image_path, conf=confidence, verbose=False)[0]

    annotated = result.plot()  # ultralyticsが描画済みの画像(BGR)を返す
    annotated_rgb = annotated[..., ::-1]  # BGR -> RGB

    fig, ax = plt.subplots(1, figsize=(8, 8))
    ax.imshow(annotated_rgb)
    ax.axis('off')
    plt.show()
    return result

# 使い方
result = detect_and_show("test_photo.jpg")